# YOLOv8-OBB Training: Bottle vs Can Detection

**Goal:** Train a YOLOv8n-OBB model on the Roboflow bottle-and-can dataset using Google Colab T4 GPU.

**Dataset:** Roboflow `bouteille/bottle-and-can` v7 — 1905 train / 486 val / 100 test images  
Train class balance: bottle 43% / can 57% (well-balanced after augmentation)

**Model:** YOLOv8n-OBB (nano, ~3M params) — oriented bounding boxes, transfer learning from COCO pretrained weights

**Training plan:** 100 epochs, imgsz=640, batch=16, AdamW optimizer, cosine LR

## 1. Environment Setup

In [ ]:
!nvidia-smi

In [ ]:
%pip install -q ultralytics roboflow
import ultralytics
ultralytics.checks()

## 2. Download Dataset from Roboflow

Downloads v7 of the `bouteille/bottle-and-can` project directly into Colab.  
No manual upload needed — the API key handles authentication.

In [ ]:
from roboflow import Roboflow
import os

rf = Roboflow(api_key="l2vJEUtQX03ZwuWPpsXi")
project = rf.workspace("bouteille").project("bottle-and-can")
dataset = project.version(7).download("yolov8-obb", location="/content/dataset")
print("Dataset location:", dataset.location)

In [ ]:
# Fix data.yaml paths for Colab absolute paths
yaml_content = """path: /content/dataset
train: train/images
val: valid/images
test: test/images

names:
  0: bottle
  1: can

nc: 2
"""
with open("/content/dataset/data.yaml", "w") as f:
    f.write(yaml_content)
print(yaml_content)

# Verify image counts
for split in ["train", "valid", "test"]:
    n = len(os.listdir(f"/content/dataset/{split}/images"))
    print(f"{split:5s}: {n} images")

## 3. Class Balance Check

In [ ]:
from collections import Counter
from pathlib import Path

root = Path("/content/dataset")
for split in ["train", "valid", "test"]:
    counts = Counter()
    for lbl in (root / split / "labels").glob("*.txt"):
        for line in lbl.read_text().strip().splitlines():
            if line:
                counts[int(line.split()[0])] += 1
    total = sum(counts.values())
    b, c = counts.get(0, 0), counts.get(1, 0)
    print(f"{split:5s}: bottle={b:4d} ({100*b/max(total,1):.0f}%)  "
          f"can={c:4d} ({100*c/max(total,1):.0f}%)  total={total}")

## 4. Sanity Check — Visualize Sample Annotations

In [ ]:
import cv2, glob, random
import numpy as np
import matplotlib.pyplot as plt

CLASS_NAMES = {0: "bottle", 1: "can"}
COLORS = {0: (0, 200, 0), 1: (0, 100, 255)}

def draw_obb(img, label_path):
    h, w = img.shape[:2]
    if not os.path.exists(label_path):
        return img
    for line in open(label_path):
        parts = line.strip().split()
        if len(parts) < 9:
            continue
        cls = int(parts[0])
        coords = list(map(float, parts[1:9]))
        pts = np.array([[coords[i]*w, coords[i+1]*h] for i in range(0, 8, 2)], dtype=np.int32)
        cv2.polylines(img, [pts], True, COLORS.get(cls, (200,200,200)), 3)
        cv2.putText(img, CLASS_NAMES.get(cls, str(cls)), tuple(pts[0]),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, COLORS.get(cls, (200,200,200)), 2)
    return img

random.seed(42)
all_imgs = glob.glob("/content/dataset/train/images/*.jpg")
sample_imgs = random.sample(all_imgs, min(6, len(all_imgs)))

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for ax, img_path in zip(axes.flat, sample_imgs):
    lbl = img_path.replace("/images/", "/labels/").replace(".jpg", ".txt")
    img = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
    img = draw_obb(img, lbl)
    ax.imshow(img)
    ax.axis("off")
plt.suptitle("Sample OBB annotations (green=bottle, orange=can)", fontsize=14)
plt.tight_layout()
plt.show()

## 5. Training

Key hyperparameter choices:
- `yolov8n-obb.pt` — nano OBB model, avoids overfitting on a ~1900-image dataset
- `epochs=100`, `patience=20` — early stopping kicks in if val mAP plateaus
- `batch=16` — fits comfortably in T4 VRAM
- `optimizer=AdamW` — more stable than SGD for small datasets
- `cos_lr=True` — cosine annealing for better final convergence
- `mosaic=1.0`, `mixup=0.15` — strong augmentation to compensate for dataset size

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n-obb.pt")

results = model.train(
    data="/content/dataset/data.yaml",
    epochs=100,
    imgsz=640,
    batch=16,
    optimizer="AdamW",
    lr0=0.001,
    cos_lr=True,
    patience=20,
    mosaic=1.0,
    mixup=0.15,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=15.0,
    translate=0.1,
    scale=0.5,
    project="/content/runs",
    name="bottle_can_obb_v2",
    exist_ok=True,
    plots=True,
)

## 6. Evaluate on Test Set

In [ ]:
best = "/content/runs/bottle_can_obb_v2/weights/best.pt"
model = YOLO(best)
metrics = model.val(data="/content/dataset/data.yaml", split="test", plots=True)

print(f"mAP@0.5      : {metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95 : {metrics.box.map:.4f}")
per_class = dict(zip(["bottle", "can"], metrics.box.maps))
print(f"bottle mAP@0.5: {per_class['bottle']:.4f}")
print(f"can    mAP@0.5: {per_class['can']:.4f}")

## 7. Save Weights and Results to Google Drive

In [ ]:
from google.colab import drive
import shutil

drive.mount("/content/drive")

DRIVE_PROJECT = "/content/drive/MyDrive/yolo-portfolio"  # change if different
os.makedirs(f"{DRIVE_PROJECT}/models", exist_ok=True)

shutil.copy(best, f"{DRIVE_PROJECT}/models/best.pt")
print("Weights saved to Drive:", f"{DRIVE_PROJECT}/models/best.pt")

if os.path.exists(f"{DRIVE_PROJECT}/results/training"):
    shutil.rmtree(f"{DRIVE_PROJECT}/results/training")
shutil.copytree("/content/runs/bottle_can_obb_v2", f"{DRIVE_PROJECT}/results/training")
print("Training results saved to Drive.")

## 8. Save Sample Predictions (for README / portfolio)

In [ ]:
import os

os.makedirs("/content/preds", exist_ok=True)

model.predict(
    source="/content/dataset/test/images",
    save=True,
    conf=0.25,
    project="/content/preds",
    name="test_predictions",
    exist_ok=True,
)

shutil.copytree(
    "/content/preds/test_predictions",
    f"{DRIVE_PROJECT}/results/sample_predictions/test",
    dirs_exist_ok=True,
)
print("Sample predictions saved to Drive.")